# MedSAM2 — ARCADE v2: Partial Unfreeze + clDice + Offline Augmentation

**Improvements over v1 (Dice 0.727):**
1. **Offline 5k augmentation** — 1000 originals × 5 versions (mild, topology-safe transforms), pre-generated once to `/home/jupyter/arcade_aug5k/` + GCS
2. **clDice loss** — topology-aware, rewards getting thin vessel centerlines right
3. **Partial encoder unfreeze** — last 2 Hiera blocks + neck adapt to X-ray

**Result:** Dice 0.767 ± 0.082 | IoU 0.629  (+0.040 vs v1)  
**Target:** Dice > 0.76 ✓  |  **Data:** ARCADE 1,000 train / 200 val

## Cell 1 — Environment Setup

In [ ]:
import os, subprocess, sys

subprocess.run(['sudo','apt-get','update','-q'], check=True)
subprocess.run(['sudo','apt-get','install','-y','ffmpeg','p7zip-full','unzip'], check=True)
subprocess.run([sys.executable,'-m','pip','install',
    'torch==2.5.1','torchvision==0.20.1',
    '--index-url','https://download.pytorch.org/whl/cu121','-q'], check=True)
subprocess.run([sys.executable,'-m','pip','install',
    'numpy>=2.0.1','opencv-python','pillow','matplotlib','tqdm','scipy',
    'pycocotools','hydra-core','omegaconf','iopath','-q'], check=True)

if not os.path.exists('/home/jupyter/MedSAM2'):
    subprocess.run(['git','clone','https://github.com/bowang-lab/MedSAM2.git',
                    '/home/jupyter/MedSAM2'], check=True)

sys.path.insert(0, '/home/jupyter/MedSAM2')
print('Environment ready')

## Cell 2 — Checkpoint + GPU

In [ ]:
import torch

BUCKET     = 'gs://YOUR_GCS_BUCKET'  # <-- set your bucket
CKPT_DIR   = '/home/jupyter/MedSAM2/checkpoints'
CHECKPOINT = os.path.join(CKPT_DIR, 'MedSAM2_latest.pt')
CONFIG     = 'configs/sam2.1_hiera_t512.yaml'
MODEL_SIZE = 512
HIRES      = MODEL_SIZE // 4
BB_FEAT_SIZES = [[HIRES//(2**k)]*2 for k in range(3)]

os.makedirs(CKPT_DIR, exist_ok=True)
if not os.path.exists(CHECKPOINT):
    subprocess.run(['bash','download.sh'], cwd='/home/jupyter/MedSAM2', check=True)
else:
    print('Checkpoint present')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0)}')

## Cell 3 — Load Data from GCS

First run: downloads from Zenodo, converts COCO → PNG masks, uploads to GCS.
Every run after: pulls silently from GCS (no Zenodo re-download).

In [ ]:
GCS_ARCADE       = f'{BUCKET}/datasets/arcade'
PROC_TRAIN_IMGS  = '/tmp/arcade_proc/train/images'
PROC_TRAIN_MASKS = '/tmp/arcade_proc/train/masks'
PROC_VAL_IMGS    = '/tmp/arcade_proc/val/images'
PROC_VAL_MASKS   = '/tmp/arcade_proc/val/masks'

def quiet_rsync(src, dst, label):
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['gsutil','-m','rsync','-r', src+'/', dst+'/'],
                   capture_output=True, check=True)
    n = len(os.listdir(dst))
    print(f'  ✓ {label} ({n} files)')

check = subprocess.run(['gsutil','ls', f'{GCS_ARCADE}/train/images/'],
                       capture_output=True, text=True)
DATA_IN_GCS = check.returncode == 0 and bool(check.stdout.strip())

if DATA_IN_GCS:
    print('Syncing ARCADE from GCS...')
    for split in ['train','val']:
        for folder in ['images','masks']:
            quiet_rsync(f'{GCS_ARCADE}/{split}/{folder}',
                        f'/tmp/arcade_proc/{split}/{folder}',
                        f'{split}/{folder}')
    print('Data ready.')
    NEED_CONVERT = False
else:
    print('No GCS cache — downloading ARCADE from Zenodo (~451 MB)...')
    ARCADE_DIR = '/tmp/arcade'
    ARCADE_ZIP = '/tmp/arcade.zip'
    if not os.path.exists(ARCADE_DIR) or not os.listdir(ARCADE_DIR):
        subprocess.run(['wget','-q','--show-progress',
            'https://zenodo.org/records/10390295/files/arcade.zip?download=1',
            '-O', ARCADE_ZIP], check=True)
        print('Extracting...')
        os.makedirs(ARCADE_DIR, exist_ok=True)
        subprocess.run(['unzip','-q', ARCADE_ZIP,'-d', ARCADE_DIR], check=True)
        os.remove(ARCADE_ZIP)
        print('Downloaded.')
    NEED_CONVERT = True

print(f'NEED_CONVERT={NEED_CONVERT}')

## Cell 4 — Convert COCO → Masks (first run only)

In [ ]:
import json as _json, numpy as np
from PIL import Image, ImageDraw
import glob
from tqdm import tqdm
from collections import defaultdict

def coco_to_binary_masks(img_dir, ann_json, out_img_dir, out_mask_dir):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)
    with open(ann_json) as f:
        coco = _json.load(f)
    id2img = {img['id']: img for img in coco['images']}
    img_anns = defaultdict(list)
    for ann in coco['annotations']:
        img_anns[ann['image_id']].append(ann)
    converted = 0
    for img_id, img_info in tqdm(id2img.items()):
        fname = img_info['file_name']
        H, W  = img_info['height'], img_info['width']
        src   = os.path.join(img_dir, fname)
        if not os.path.exists(src): continue
        stem = os.path.splitext(fname)[0]
        Image.open(src).convert('RGB').save(os.path.join(out_img_dir, f'{stem}.png'))
        mask = Image.new('L', (W, H), 0)
        draw = ImageDraw.Draw(mask)
        for ann in img_anns[img_id]:
            for seg in ann.get('segmentation', []):
                if len(seg) >= 6:
                    draw.polygon([(seg[i], seg[i+1]) for i in range(0, len(seg)-1, 2)], fill=255)
        mask.save(os.path.join(out_mask_dir, f'{stem}.png'))
        converted += 1
    return converted

if NEED_CONVERT:
    # Find SYNTAX_DIR
    SYNTAX_DIR = None
    for candidate in [os.path.join(ARCADE_DIR,'arcade','syntax'), os.path.join(ARCADE_DIR,'syntax')]:
        if os.path.exists(candidate): SYNTAX_DIR = candidate; break
    if SYNTAX_DIR is None:
        for root,dirs,_ in os.walk(ARCADE_DIR):
            if 'syntax' in dirs: SYNTAX_DIR = os.path.join(root,'syntax'); break

    def find_json(base, names):
        for n in names:
            p = os.path.join(base, n)
            if os.path.exists(p): return p
        for root,_,files in os.walk(base):
            for f in files:
                if f.endswith('.json'): return os.path.join(root, f)

    print('Converting train...')
    n = coco_to_binary_masks(
        os.path.join(SYNTAX_DIR,'train','images'),
        find_json(os.path.join(SYNTAX_DIR,'train'), ['annotations/train.json','annotations.json']),
        PROC_TRAIN_IMGS, PROC_TRAIN_MASKS)
    print(f'  {n} train images')

    print('Converting val...')
    n = coco_to_binary_masks(
        os.path.join(SYNTAX_DIR,'val','images'),
        find_json(os.path.join(SYNTAX_DIR,'val'), ['annotations/val.json','annotations.json']),
        PROC_VAL_IMGS, PROC_VAL_MASKS)
    print(f'  {n} val images')

    print('Uploading to GCS...')
    for split in ['train','val']:
        for folder in ['images','masks']:
            subprocess.run(['gsutil','-m','rsync','-r',
                f'/tmp/arcade_proc/{split}/{folder}/',
                f'{GCS_ARCADE}/{split}/{folder}/'],
                capture_output=True, check=True)
            print(f'  ✓ uploaded {split}/{folder}')
    print('Done — future runs load from GCS.')

## Cell 4.5 — Generate 5k Offline Augmented Images

**Why offline instead of online:**  
- Workers just read PNGs — no transform pipelines running in DataLoader workers  
- Deterministic and reproducible — same 5 variants every run  
- Stored at `/home/jupyter/arcade_aug5k/` — survives kernel restarts

**5 variants per image (1000 × 5 = 5000 total):**

| Suffix | Transform |
|--------|-----------|
| `_orig` | Original |
| `_hf` | Horizontal flip |
| `_vf` | Vertical flip |
| `_r20` | 20° CW rotation |
| `_r20_hf` | 20° CW + horizontal flip |

Image and mask receive **identical** geometric transforms. Masks use nearest-neighbour interpolation to preserve hard binary edges.

In [ ]:
import shutil
from tqdm import tqdm

AUG_LOCAL       = '/home/jupyter/arcade_aug5k'
AUG_TRAIN_IMGS  = f'{AUG_LOCAL}/train/images'
AUG_TRAIN_MASKS = f'{AUG_LOCAL}/train/masks'
SUFFIXES        = ['orig', 'hf', 'vf', 'r20', 'r20_hf']   # 1000 × 5 = 5000 total

# Source originals — defined here because Cell 5 (verification) comes after
_src_img_paths  = sorted(glob.glob(f'{PROC_TRAIN_IMGS}/*.png'))
_src_mask_paths = sorted(glob.glob(f'{PROC_TRAIN_MASKS}/*.png'))
assert len(_src_img_paths) > 0, 'No source images found — run Cells 3 and 4 first to populate PROC_TRAIN_IMGS'

def augment_pair(img_pil, msk_pil):
    """Return 5 (img, mask) PIL pairs — image and mask transformed identically."""
    pairs = []
    # orig
    pairs.append((img_pil, msk_pil))
    # horizontal flip
    pairs.append((img_pil.transpose(Image.FLIP_LEFT_RIGHT),
                  msk_pil.transpose(Image.FLIP_LEFT_RIGHT)))
    # vertical flip
    pairs.append((img_pil.transpose(Image.FLIP_TOP_BOTTOM),
                  msk_pil.transpose(Image.FLIP_TOP_BOTTOM)))
    # 20° CW rotation — no expand, crops corners (matches X-ray circular field of view)
    pairs.append((img_pil.rotate(-20, resample=Image.BICUBIC),
                  msk_pil.rotate(-20, resample=Image.NEAREST)))
    # horizontal flip + 20° CW rotation
    img_hf = img_pil.transpose(Image.FLIP_LEFT_RIGHT)
    msk_hf = msk_pil.transpose(Image.FLIP_LEFT_RIGHT)
    pairs.append((img_hf.rotate(-20, resample=Image.BICUBIC),
                  msk_hf.rotate(-20, resample=Image.NEAREST)))
    return pairs   # len == 5

# Check if already generated with current naming scheme
_stem0  = os.path.splitext(os.path.basename(_src_img_paths[0]))[0]
_sample = os.path.join(AUG_TRAIN_IMGS, f'{_stem0}_orig.png')

if os.path.exists(_sample):
    print(f'Aug5k already present — skipping generation')
else:
    if os.path.exists(AUG_LOCAL):
        shutil.rmtree(AUG_LOCAL)   # clear any old naming scheme (e.g. _v0/_v4 from albumentations era)
    os.makedirs(AUG_TRAIN_IMGS,  exist_ok=True)
    os.makedirs(AUG_TRAIN_MASKS, exist_ok=True)

    for img_path, msk_path in tqdm(zip(_src_img_paths, _src_mask_paths),
                                   total=len(_src_img_paths), desc='Augmenting'):
        stem    = os.path.splitext(os.path.basename(img_path))[0]
        img_pil = Image.open(img_path).convert('RGB')
        msk_pil = Image.open(msk_path).convert('L')

        for suffix, (aug_img, aug_msk) in zip(SUFFIXES, augment_pair(img_pil, msk_pil)):
            fname = f'{stem}_{suffix}.png'
            aug_img.save(os.path.join(AUG_TRAIN_IMGS, fname))
            msk_bin = (np.array(aug_msk) > 0).astype(np.uint8) * 255
            Image.fromarray(msk_bin).save(os.path.join(AUG_TRAIN_MASKS, fname))

    print(f'Generated {len(os.listdir(AUG_TRAIN_IMGS))} images → {AUG_LOCAL}')

# Construct paths from stems + suffixes — avoids picking up stale files from old runs
aug_img_paths  = [os.path.join(AUG_TRAIN_IMGS,  f'{os.path.splitext(os.path.basename(p))[0]}_{s}.png')
                  for p in _src_img_paths for s in SUFFIXES]
aug_mask_paths = [os.path.join(AUG_TRAIN_MASKS, f'{os.path.splitext(os.path.basename(p))[0]}_{s}.png')
                  for p in _src_img_paths for s in SUFFIXES]
print(f'Aug5k ready: {len(aug_img_paths)} images | {len(aug_mask_paths)} masks')

In [ ]:
import matplotlib.pyplot as plt

# Verify augmentation quality — all 5 variants of one image, with mask overlay
_stem_check = os.path.splitext(os.path.basename(_src_img_paths[0]))[0]
fig, axes = plt.subplots(2, len(SUFFIXES), figsize=(4 * len(SUFFIXES), 8))
for col, suffix in enumerate(SUFFIXES):
    img_v = np.array(Image.open(f'{AUG_TRAIN_IMGS}/{_stem_check}_{suffix}.png').convert('L'))
    msk_v = np.array(Image.open(f'{AUG_TRAIN_MASKS}/{_stem_check}_{suffix}.png'))
    axes[0, col].imshow(img_v, cmap='gray'); axes[0, col].set_title(suffix); axes[0, col].axis('off')
    axes[1, col].imshow(img_v, cmap='gray')
    axes[1, col].imshow(msk_v > 0, alpha=0.4, cmap='Reds')
    axes[1, col].axis('off')
axes[0, 0].set_ylabel('Image', fontsize=11)
axes[1, 0].set_ylabel('Image + Mask', fontsize=11)
plt.suptitle(f'Augmentation verification — {_stem_check}', fontsize=13)
plt.tight_layout(); plt.show()

## Cell 5 — Verify Pairs + Spot Check

In [ ]:
train_img_paths  = sorted(glob.glob(f'{PROC_TRAIN_IMGS}/*.png'))
train_mask_paths = sorted(glob.glob(f'{PROC_TRAIN_MASKS}/*.png'))
test_img_paths   = sorted(glob.glob(f'{PROC_VAL_IMGS}/*.png'))
test_mask_paths  = sorted(glob.glob(f'{PROC_VAL_MASKS}/*.png'))
print(f'Train: {len(train_img_paths)} | Test: {len(test_img_paths)}')

fg_list = [(np.array(Image.open(mp).convert('L').resize((256,256),Image.NEAREST))>0).mean()
           for mp in train_mask_paths[:50]]
print(f'Foreground fraction: {np.mean(fg_list)*100:.1f}%')

fig, axes = plt.subplots(2,4,figsize=(18,8))
for col in range(4):
    idx = col*(len(train_img_paths)//4)
    img = np.array(Image.open(train_img_paths[idx]).convert('L'))
    msk = np.array(Image.open(train_mask_paths[idx]).convert('L'))
    axes[0,col].imshow(img,cmap='gray'); axes[0,col].axis('off')
    axes[0,col].set_title(os.path.basename(train_img_paths[idx]))
    axes[1,col].imshow(img,cmap='gray'); axes[1,col].imshow(msk>0,alpha=0.4,cmap='Reds')
    axes[1,col].axis('off')
axes[0,0].set_ylabel('Image'); axes[1,0].set_ylabel('+ Mask')
plt.suptitle('ARCADE spot check — verify vessel masks', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
def centroid_click(mask_np):
    ys, xs = np.where(mask_np > 0)
    if len(ys) == 0: return (mask_np.shape[1]//2, mask_np.shape[0]//2)
    return (int(xs.mean()), int(ys.mean()))

def dice_score(pred, gt, smooth=1e-5):
    p, g = pred.astype(float), gt.astype(float)
    return (2*(p*g).sum()+smooth) / (p.sum()+g.sum()+smooth)

def iou_score(pred, gt, smooth=1e-5):
    i = (pred.astype(bool) & gt.astype(bool)).sum()
    u = (pred.astype(bool) | gt.astype(bool)).sum()
    return (i+smooth) / (u+smooth)

print('Helpers ready')

## Cell 6 — Zero-Shot Baseline (200 val images)

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

zs_model     = build_sam2(CONFIG, CHECKPOINT, device=device)
zs_predictor = SAM2ImagePredictor(zs_model)

zs_results = []
for img_path, mask_path in tqdm(zip(test_img_paths, test_mask_paths), total=len(test_img_paths), desc='Zero-shot'):
    img_rgb = np.array(Image.open(img_path).convert('RGB'))
    gt_mask = np.array(Image.open(mask_path).convert('L')) > 0
    cx, cy  = centroid_click(gt_mask)
    with torch.no_grad():
        zs_predictor.set_image(img_rgb)
        preds, _, _ = zs_predictor.predict(
            point_coords=np.array([[cx,cy]]), point_labels=np.array([1]),
            multimask_output=False)
    zs_results.append({'dice': dice_score(preds[0], gt_mask),
                       'iou':  iou_score(preds[0], gt_mask),
                       'pred': preds[0], 'gt': gt_mask, 'img': img_rgb, 'path': img_path})

zs_dice = [r['dice'] for r in zs_results]
zs_iou  = [r['iou']  for r in zs_results]
print(f'Zero-Shot | Dice: {np.mean(zs_dice):.3f} ± {np.std(zs_dice):.3f} | IoU: {np.mean(zs_iou):.3f}')

## Cell 7 — Dataset + DataLoader

Reads from the 5k offline augmented dataset generated in Cell 4.5.
Workers only read PNGs — no online transform pipeline in the DataLoader.

In [ ]:
import torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'

class ArcadeDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.transform  = transforms.Compose([
            transforms.Resize((MODEL_SIZE, MODEL_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        img      = Image.open(self.img_paths[idx]).convert('RGB')
        mask_pil = Image.open(self.mask_paths[idx]).convert('L')
        mask_np  = np.array(mask_pil)
        img_t    = self.transform(img)
        mask_256 = np.array(mask_pil.resize((256, 256), Image.NEAREST))
        mask_t   = torch.from_numpy((mask_256 > 0).astype(np.float32)).unsqueeze(0)
        ys, xs   = np.where(mask_np > 0)
        if len(ys) > 0:
            pick = np.random.randint(len(ys))
            py, px = int(ys[pick]), int(xs[pick])
        else:
            py, px = mask_np.shape[0]//2, mask_np.shape[1]//2
        h, w  = mask_np.shape
        point = torch.tensor([px/w*MODEL_SIZE, py/h*MODEL_SIZE], dtype=torch.float32)
        return img_t, mask_t, point

fg_list    = [(np.array(Image.open(mp).convert('L').resize((256,256), Image.NEAREST)) > 0).mean()
              for mp in aug_mask_paths[:200]]
fg_mean    = float(np.mean(fg_list))
pos_weight = torch.tensor([(1 - fg_mean) / fg_mean], device=device)
print(f'Foreground: {fg_mean*100:.1f}%  |  pos_weight: {pos_weight.item():.1f}x')

train_ds     = ArcadeDataset(aug_img_paths, aug_mask_paths)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} images | {len(train_loader)} batches/epoch')

## Cell 8 — clDice Loss

**What clDice is (plain English):**
Regular Dice treats every pixel equally — a wide vessel and a thin distal branch count the same.
clDice first extracts the *skeleton* (centerline) of each vessel, then checks if the prediction
and ground truth skeletons match. This forces the model to capture thin branches that Dice alone ignores.

**Warmup:** clDice is ramped in over epochs 3–8. On a fresh model, the skeleton of a bad prediction
is unstable and clDice loss would thrash the training. Warmup prevents this.

In [ ]:
import torch.utils.checkpoint as cp

def soft_erode(x):
    p1 = -F.max_pool2d(-x, (3,1), (1,1), (1,0))
    p2 = -F.max_pool2d(-x, (1,3), (1,1), (0,1))
    return torch.min(p1, p2)

def soft_dilate(x):
    return F.max_pool2d(x, (3,3), (1,1), (1,1))

def soft_open(x):
    return soft_dilate(soft_erode(x))

def _skel_inner(x):
    # fp32 skeleton — called via checkpoint so intermediate tensors aren't stored
    x = x.float()
    skel = F.relu(x - soft_open(x))
    for _ in range(10):
        x    = soft_erode(x)
        delta = F.relu(x - soft_open(x))
        skel  = skel + F.relu(delta - skel * delta)
    return skel

def soft_skel(x):
    # gradient checkpointing: recomputes the 10-iter loop on backward
    # instead of storing all intermediate tensors — halves peak memory, same result
    return cp.checkpoint(_skel_inner, x, use_reentrant=False)

def soft_cldice_loss(probs, target, smooth=1.0):
    # probs and target must already be float32 sigmoid outputs
    skel_pred = soft_skel(probs)
    with torch.no_grad():
        skel_true = _skel_inner(target)   # GT skeleton: no grad needed, skip checkpoint overhead
    tprec = (torch.sum(skel_pred * target) + smooth) / (torch.sum(skel_pred) + smooth)
    tsens = (torch.sum(skel_true * probs)  + smooth) / (torch.sum(skel_true) + smooth)
    return 1.0 - 2.0*(tprec*tsens)/(tprec+tsens)

def dice_loss(logits, target, smooth=1e-5):
    pred  = torch.sigmoid(logits)
    inter = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    return (1 - (2*inter+smooth)/(union+smooth)).mean()

def combined_loss(logits, target, pw, cldice_w):
    # Dice + wBCE run in fp16 (inside autocast)
    d = dice_loss(logits, target)
    b = nn.BCEWithLogitsLoss(pos_weight=pw)(logits, target)

    # clDice runs outside autocast in fp32 — explicit boundary avoids hidden dtype casts
    if cldice_w > 0:
        with torch.autocast(device_type='cuda', enabled=False):
            probs = torch.sigmoid(logits.float())
            cl    = soft_cldice_loss(probs, target.float())
    else:
        cl = torch.tensor(0.0, device=logits.device)

    return 0.5*d + 0.2*b + cldice_w*cl, d.item(), cl.item()

print('clDice loss ready (gradient checkpointed)')

## Cell 9 — Model: Partial Encoder Unfreeze

**What's unfrozen:**
- `image_encoder.trunk.blocks[10]` and `[11]` — last 2 of 12 Hiera blocks (deepest, most semantic)
- `image_encoder.neck` — the FPN that bridges encoder → decoder

**Why not unfreeze more?**
Stage 3 of Hiera has 7 blocks (indices 3–9). Unfreezing those on 1,000 images risks
catastrophic forgetting — overwriting the general features the encoder learned from millions
of CT/MRI images. The last 2 blocks are the safest: highest-level features, smallest parameter count.

**Discriminative learning rates:**
- Decoder: 5e-5 (same as before)
- Neck: 1e-5 (0.2×)
- Unfrozen trunk blocks: 5e-6 (0.1×)
Standard ratio used in SAM-Med2D and MedSAM fine-tuning literature.

In [ ]:
model = build_sam2(CONFIG, CHECKPOINT, device=device)

UNFROZEN_BLOCKS = {"blocks.10", "blocks.11"}

def build_param_groups(model, base_lr=5e-5):
    decoder_p, neck_p, trunk_p = [], [], []
    for name, p in model.named_parameters():
        if "image_encoder.trunk" in name:
            if any(b in name for b in UNFROZEN_BLOCKS):
                p.requires_grad = True
                trunk_p.append(p)
            else:
                p.requires_grad = False
        elif "image_encoder.neck" in name:
            p.requires_grad = True
            neck_p.append(p)
        else:
            p.requires_grad = True
            decoder_p.append(p)

    def split_wd(params):
        decay, no_decay = [], []
        for p in params:
            (no_decay if p.ndim <= 1 else decay).append(p)
        return decay, no_decay

    d_dec, nd_dec = split_wd(decoder_p)
    d_neck, nd_neck = split_wd(neck_p)
    d_trunk, nd_trunk = split_wd(trunk_p)
    return [
        {"params": d_dec,    "lr": base_lr,        "weight_decay": 0.01},
        {"params": nd_dec,   "lr": base_lr,        "weight_decay": 0.0},
        {"params": d_neck,   "lr": base_lr*0.2,    "weight_decay": 0.01},
        {"params": nd_neck,  "lr": base_lr*0.2,    "weight_decay": 0.0},
        {"params": d_trunk,  "lr": base_lr*0.1,    "weight_decay": 0.01},
        {"params": nd_trunk, "lr": base_lr*0.1,    "weight_decay": 0.0},
    ]

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')
print(f'Decoder LR: 5e-5 | Neck LR: 1e-5 | Trunk (last 2 blocks) LR: 5e-6')

## Cell 10 — Training (20 epochs)

20 epochs over 5k offline-augmented images (1000 originals × 5 PIL variants).
clDice warmup: 0 for epochs 1–3, ramps to 0.3 by epoch 8.
Checkpoint saved to disk and backed up to GCS after every best epoch.

In [ ]:
import threading, queue

CKPT_PATH = '/home/jupyter/medsam2_arcade_v2.pt'
GCS_CKPT  = f'{BUCKET}/checkpoints/medsam2_arcade_v2.pt'
EPOCHS    = 20
BASE_LR   = 5e-5

optimizer = optim.AdamW(build_param_groups(model, BASE_LR), betas=(0.9, 0.999))
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler()
best_loss = float('inf')
train_losses = []

# Background uploader — no zombie processes, errors surface
_upload_q = queue.Queue()

def _uploader():
    while True:
        item = _upload_q.get()
        if item is None:
            return
        local, remote = item
        try:
            subprocess.run(['gsutil', '-q', 'cp', local, remote],
                           check=True, timeout=600, capture_output=True)
        except Exception as e:
            print(f'[uploader] warn: {e}')
        finally:
            _upload_q.task_done()

threading.Thread(target=_uploader, daemon=True).start()

def gcs_upload_async(local, remote):
    _upload_q.put((local, remote))

def set_train_mode(model):
    model.train()
    model.image_encoder.trunk.eval()
    model.image_encoder.trunk.blocks[10].train()
    model.image_encoder.trunk.blocks[11].train()
    model.image_encoder.neck.train()

print(f'Training {EPOCHS} epochs | fp16 autocast | clDice checkpointed | thread-queue GCS save')
print(f'Dataset: {len(train_ds)} images | {len(train_loader)} batches/epoch')

for epoch in range(EPOCHS):
    set_train_mode(model)
    # clDice warmup: 0 for epochs 1-3, ramps to 0.3 by epoch 8
    cldice_w = min(0.3, 0.3 * max(0, epoch - 3) / 5)
    ep_loss = ep_dice = ep_cldice = 0.0

    for imgs, masks, points in tqdm(train_loader, desc=f'Ep {epoch+1}/{EPOCHS}  clDice_w={cldice_w:.2f}', leave=False):
        optimizer.zero_grad(set_to_none=True)
        B = imgs.shape[0]
        imgs_dev, points_dev = imgs.to(device), points.to(device)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            backbone_out = model.forward_image(imgs_dev)
            _, vision_feats, _, _ = model._prepare_backbone_features(backbone_out)
            if model.directly_add_no_mem_embed:
                vision_feats[-1] = vision_feats[-1] + model.no_mem_embed
            feats = [feat.permute(1,2,0).view(B,-1,*fs)
                     for feat, fs in zip(vision_feats[::-1], BB_FEAT_SIZES[::-1])][::-1]
            image_embed, high_res_feats = feats[-1], feats[:-1]

            all_logits = []
            for i in range(B):
                pt       = points_dev[i].reshape(1,1,2)
                pt_label = torch.ones(1, 1, dtype=torch.int, device=device)
                sparse_emb, dense_emb = model.sam_prompt_encoder(
                    points=(pt, pt_label), boxes=None, masks=None)
                low_res_masks, _, _, _ = model.sam_mask_decoder(
                    image_embeddings=image_embed[i].unsqueeze(0),
                    image_pe=model.sam_prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_emb,
                    dense_prompt_embeddings=dense_emb,
                    multimask_output=False, repeat_image=False,
                    high_res_features=[f[i].unsqueeze(0) for f in high_res_feats])
                all_logits.append(low_res_masks)

            logits  = torch.cat(all_logits, dim=0)
            targets = masks.to(device)
            if targets.shape[-2:] != logits.shape[-2:]:
                targets = F.interpolate(targets.float(), size=logits.shape[-2:], mode='nearest')
            loss, d_val, cl_val = combined_loss(logits, targets, pos_weight, cldice_w)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if torch.isfinite(loss):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], max_norm=0.5)
            scaler.step(optimizer)
        scaler.update()
        ep_loss   += loss.item()
        ep_dice   += d_val
        ep_cldice += cl_val

    n = len(train_loader)
    ep_loss /= n; ep_dice /= n; ep_cldice /= n
    train_losses.append(ep_loss)
    scheduler.step()
    print(f'Ep {epoch+1:2d} | Loss: {ep_loss:.4f}  Dice: {ep_dice:.4f}  clDice: {ep_cldice:.4f}')

    if ep_loss < best_loss:
        best_loss = ep_loss
        torch.save(model.state_dict(), CKPT_PATH)
        gcs_upload_async(CKPT_PATH, GCS_CKPT)
        print(f'           ★ saved → /home/jupyter/ + GCS (queued)')

print(f'\nBest loss: {best_loss:.4f}')

plt.figure(figsize=(8,4))
plt.plot(range(1, len(train_losses)+1), train_losses, 'b-o', markersize=4)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('ARCADE v2 — partial unfreeze + clDice + 5k offline aug (20 epochs)')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/home/jupyter/arcade_v2_loss.png', dpi=100); plt.show()
gcs_upload_async('/home/jupyter/arcade_v2_loss.png', f'{BUCKET}/results/arcade_v2_loss.png')
print('Loss curve → /home/jupyter/ + GCS (queued)')

## Cell 11 — Fine-Tuned Evaluation (200 val images)

In [ ]:
ft_model = build_sam2(CONFIG, CHECKPOINT, device=device)

if not os.path.exists(CKPT_PATH):
    print('Pulling checkpoint from GCS...')
    subprocess.run(['gsutil','cp', GCS_CKPT, CKPT_PATH], check=True)

incomp = ft_model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=False), strict=False)
print(f'Missing: {len(incomp.missing_keys)} | Unexpected: {len(incomp.unexpected_keys)}')
assert len(incomp.missing_keys) == 0, '⚠ Checkpoint load failed!'
print('✓ Clean checkpoint load')
ft_model.eval()
ft_predictor = SAM2ImagePredictor(ft_model)

ft_results = []
for img_path, mask_path in tqdm(zip(test_img_paths, test_mask_paths),
                                 total=len(test_img_paths), desc='Fine-tuned eval'):
    img_rgb = np.array(Image.open(img_path).convert('RGB'))
    gt_mask = np.array(Image.open(mask_path).convert('L')) > 0
    cx, cy  = centroid_click(gt_mask)
    with torch.no_grad():
        ft_predictor.set_image(img_rgb)
        preds, _, _ = ft_predictor.predict(
            point_coords=np.array([[cx,cy]]), point_labels=np.array([1]),
            multimask_output=False)
    ft_results.append({'dice': dice_score(preds[0], gt_mask),
                       'iou':  iou_score(preds[0], gt_mask),
                       'pred': preds[0], 'gt': gt_mask, 'img': img_rgb})

ft_dice = [r['dice'] for r in ft_results]
ft_iou  = [r['iou']  for r in ft_results]
print(f'\n=== ARCADE v2 Results (200 val images) ===')
print(f'Zero-Shot      | Dice: {np.mean(zs_dice):.3f} ± {np.std(zs_dice):.3f} | IoU: {np.mean(zs_iou):.3f}')
print(f'v1 (frozen)    | Dice: 0.727 ± 0.081 | IoU: 0.577  [reference]')
print(f'v2 (this run)  | Dice: {np.mean(ft_dice):.3f} ± {np.std(ft_dice):.3f} | IoU: {np.mean(ft_iou):.3f}')
print(f'Delta vs v1:   | {np.mean(ft_dice)-0.727:+.3f} Dice')

In [ ]:
ft_sorted = sorted(enumerate(ft_results), key=lambda x: x[1]['dice'])
show_idx  = [ft_sorted[0][0], ft_sorted[len(ft_sorted)//4][0],
             ft_sorted[len(ft_sorted)//2][0], ft_sorted[3*len(ft_sorted)//4][0],
             ft_sorted[-2][0], ft_sorted[-1][0]]

fig, axes = plt.subplots(3,6,figsize=(26,11))
for col, idx in enumerate(show_idx):
    fr = ft_results[idx]; zr = zs_results[idx]
    img = fr['img'][:,:,0]
    axes[0,col].imshow(img,cmap='gray'); axes[0,col].imshow(fr['gt'],alpha=0.35,cmap='Greens')
    axes[0,col].set_title(f'GT D={fr["dice"]:.2f}'); axes[0,col].axis('off')
    axes[1,col].imshow(img,cmap='gray'); axes[1,col].imshow(zr['pred'],alpha=0.45,cmap='Reds')
    axes[1,col].set_title(f'ZS D={zr["dice"]:.2f}'); axes[1,col].axis('off')
    axes[2,col].imshow(img,cmap='gray'); axes[2,col].imshow(fr['pred'],alpha=0.45,cmap='Blues')
    axes[2,col].set_title(f'FT D={fr["dice"]:.2f}'); axes[2,col].axis('off')
for row, label in enumerate(['GT overlay','Zero-Shot','v2 Fine-Tuned']):
    axes[row,0].set_ylabel(label, fontsize=10)
plt.suptitle(f'ARCADE v2 | ZS={np.mean(zs_dice):.3f}  FT={np.mean(ft_dice):.3f} '
             f'(v1=0.727, Δ={np.mean(ft_dice)-0.727:+.3f})', fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/arcade_v2_comparison.png', dpi=120, bbox_inches='tight'); plt.show()
subprocess.run(['gsutil','cp','/tmp/arcade_v2_comparison.png',
                f'{BUCKET}/results/arcade_v2_comparison.png'],
               check=True, capture_output=True)
print('Comparison panel → GCS')